In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251117_145413"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "10_dashboard_results"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2. Import libs
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Exemplo de resultados (em prática, salvaríamos de cada notebook em JSON/CSV)
results = {
    "Baseline (Logit)": {"AUC": 0.55, "MDA": 0.52},
    "TF-IDF (Logit)": {"AUC": 0.66, "MDA": 0.58},
    "TF-IDF (XGBoost)": {"AUC": 0.50, "MDA": 0.58},
    "Embeddings (Logit)": {"AUC": 0.56, "MDA": 0.58},
    "Embeddings (RandomForest)": {"AUC": 0.44, "MDA": 0.66},
    "LSTM": {"AUC": 0.60, "MDA": 0.62}
}

df_res = pd.DataFrame(results).T.reset_index().rename(columns={"index":"Modelo"})
df_res

,Modelo,AUC,MDA
0,Baseline (Logit),0.55,0.52
1,TF-IDF (Logit),0.66,0.58
2,TF-IDF (XGBoost),0.50,0.58
3,Embeddings (Logit),0.56,0.58
4,Embeddings (RandomForest),0.44,0.66
5,LSTM,0.60,0.62


In [4]:
# 3. Tabela comparativa
fig_table = go.Figure(data=[go.Table(
    header=dict(values=list(df_res.columns), fill_color='lightblue', align='center'),
    cells=dict(values=[df_res[col] for col in df_res.columns], align='center')
)])
fig_table.update_layout(title="Comparação de Modelos - AUC e MDA")
fig_table.show()

In [5]:
# 4. Gráfico de barras comparando AUC e MDA
fig_bar = go.Figure()

fig_bar.add_trace(go.Bar(
    x=df_res["Modelo"], y=df_res["AUC"], name="AUC", marker_color="steelblue"
))
fig_bar.add_trace(go.Bar(
    x=df_res["Modelo"], y=df_res["MDA"], name="MDA", marker_color="darkorange"
))

fig_bar.update_layout(
    barmode="group",
    title="Desempenho dos Modelos (AUC vs MDA)",
    xaxis_title="Modelo",
    yaxis_title="Score"
)
fig_bar.show()

In [6]:
# 5. Visualização do Ibovespa com marcação de notícias
ibov_file = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_file = os.path.join(PROC_PATH, "noticias_real_clean.csv")

df_ibov = pd.read_csv(ibov_file)
df_news = pd.read_csv(news_file)

if "date" in df_ibov.columns: df_ibov.rename(columns={"date":"data"}, inplace=True)
if "date" in df_news.columns: df_news.rename(columns={"date":"data"}, inplace=True)

df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

fig_ibov = go.Figure()
fig_ibov.add_trace(go.Scatter(
    x=df_ibov["data"], y=df_ibov["close"],
    mode="lines", name="IBOV", line=dict(color="blue")
))
fig_ibov.add_trace(go.Scatter(
    x=df_news["data"], y=[df_ibov["close"].iloc[-1]]*len(df_news),
    mode="markers", name="Notícias",
    text=df_news["title"] if "title" in df_news.columns else df_news["clean_text"],
    marker=dict(color="red", size=8, symbol="circle")
))

fig_ibov.update_layout(title="IBOV com Marcação de Notícias", xaxis_title="Data", yaxis_title="Fechamento")
fig_ibov.show()